In [12]:
import sqlite3
from sqlite3 import Error


## SQLite-backed API

The `api.py` script defines a FastAPI application that connects to the SQLite database created earlier. It exposes CRUD endpoints for `properties`, `units`, and `residents`.

Install dependencies:
```bash
pip install -r requirements.txt
```

Run the server (choose one):
```bash
# Start with empty database
uvicorn api:app --reload --host 0.0.0.0 --port 8000

# Start with auto-generated fake data (5 properties, 5-15 units each)
LOAD_FAKE_DATA=true uvicorn api:app --reload --host 0.0.0.0 --port 8000

# Start with custom data volume
LOAD_FAKE_DATA=true NUM_PROPERTIES=10 MIN_UNITS_PER_PROPERTY=10 MAX_UNITS_PER_PROPERTY=30 RESIDENTS_RATIO=0.8 uvicorn api:app --reload
```

Generate test data independently:
```bash
python seed_data.py  # Generates default data
```

Run tests:
```bash
pytest test_api.py
```

You can then interact with the database via HTTP requests or by visiting the interactive docs at `http://localhost:8000/docs`.

In [13]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
        print("Connection to SQLite DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")

    return connection

In [14]:
connection = create_connection("welltower.db")

Connection to SQLite DB successful


In [15]:
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as e:
        print(f"The error '{e}' occurred")

In [16]:
create_property_table = """
CREATE TABLE IF NOT EXISTS properties (
  property_id INT PRIMARY KEY NOT NULL,
  property_name TEXT NOT NULL,
  owner TEXT NOT NULL
);
"""
execute_query(connection, create_property_table)  


Query executed successfully


In [17]:
create_units_table = """
CREATE TABLE IF NOT EXISTS units (
  unit_id INT PRIMARY KEY NOT NULL,
  unit_number TEXT NOT NULL,
  property_id INT NOT NULL,
  unit_status TEXT NOT NULL CHECK(unit_status IN ('active', 'inactive')),
  occupied BOOLEAN NOT NULL,
  rent FLOAT NOT NULL,
  FOREIGN KEY (property_id) REFERENCES properties(property_id)
);
"""
execute_query(connection, create_units_table)  


Query executed successfully


In [18]:
create_residents_table = """
CREATE TABLE IF NOT EXISTS residents (
  resident_id INT PRIMARY KEY NOT NULL,
  unit_id INT NOT NULL,
  property_id INT NOT NULL,
  first_name TEXT NOT NULL,
  last_name TEXT NOT NULL,
  move_in_date DATE NOT NULL,
  move_out_date DATE,
  FOREIGN KEY (unit_id) REFERENCES units(unit_id),
  FOREIGN KEY (property_id) REFERENCES properties(property_id)
);
"""
execute_query(connection, create_residents_table)

Query executed successfully


In [19]:
create_rentroll_table = """
CREATE TABLE IF NOT EXISTS rentroll (
  rentroll_id INT PRIMARY KEY NOT NULL,
  date DATE NOT NULL,
  property_id INT NOT NULL,
  unit_id INT NOT NULL,
  unit_number TEXT NOT NULL,
  resident_id INT,
  resident_name TEXT,
  rent_amount FLOAT NOT NULL,
  unit_status TEXT NOT NULL CHECK(unit_status IN ('active', 'inactive')),
  FOREIGN KEY (unit_id) REFERENCES units(unit_id),
  FOREIGN KEY (property_id) REFERENCES properties(property_id),
  FOREIGN KEY (resident_id) REFERENCES residents(resident_id)
);
"""
execute_query(connection, create_rentroll_table)

Query executed successfully


In [20]:
create_scd_table = """
CREATE TABLE IF NOT EXISTS scd (
  scd_id INTEGER PRIMARY KEY AUTOINCREMENT,
  date DATE NOT NULL,
  property_id INTEGER,
  unit_id INTEGER,
  resident_id INTEGER,
  property_name TEXT,
  owner TEXT,
  unit_number TEXT,
  unit_status TEXT,
  occupied BOOLEAN,
  rent REAL,
  first_name TEXT,
  last_name TEXT,
  effective_from DATE,
  effective_to DATE,
  current_flag BOOLEAN DEFAULT 1,
  FOREIGN KEY (property_id) REFERENCES properties(property_id),
  FOREIGN KEY (unit_id) REFERENCES units(unit_id),
  FOREIGN KEY (resident_id) REFERENCES residents(resident_id)
);
"""
execute_query(connection, create_scd_table)

Query executed successfully


In [21]:
# Insert fake data into properties
insert_properties = """
INSERT INTO properties (property_id, property_name, owner) VALUES
(1, 'Sunset Apartments', 'Welltower Inc.'),
(2, 'River View Towers', 'Welltower Inc.'),
(3, 'Mountain Ridge Estates', 'Welltower Inc.');
"""
execute_query(connection, insert_properties)

# Insert fake data into units
insert_units = """
INSERT INTO units (unit_id, unit_number, property_id, unit_status, occupied, rent) VALUES
(1, '101', 1, 'active', 1, 1200.00),
(2, '102', 1, 'active', 1, 1300.00),
(3, '201', 1, 'active', 0, 1250.00),
(4, '101', 2, 'active', 1, 1500.00),
(5, '102', 2, 'inactive', 0, 1400.00),
(6, '101', 3, 'active', 1, 1600.00);
"""
execute_query(connection, insert_units)

# Insert fake data into residents
insert_residents = """
INSERT INTO residents (resident_id, unit_id, property_id, first_name, last_name, move_in_date, move_out_date) VALUES
(1, 1, 1, 'John', 'Doe', '2023-01-01', NULL),
(2, 2, 1, 'Jane', 'Smith', '2023-02-01', '2024-02-01'),
(3, 4, 2, 'Alice', 'Johnson', '2023-03-01', NULL),
(4, 6, 3, 'Bob', 'Brown', '2023-04-01', NULL);
"""
execute_query(connection, insert_residents)

The error 'UNIQUE constraint failed: properties.property_id' occurred
The error 'UNIQUE constraint failed: units.unit_id' occurred
The error 'UNIQUE constraint failed: residents.resident_id' occurred


In [22]:
# Populate SCD table with resident data
populate_scd_residents = """
INSERT INTO scd (property_id, unit_id, resident_id, property_name, owner, unit_number, unit_status, occupied, rent, first_name, last_name, effective_from, effective_to, current_flag)
SELECT p.property_id, u.unit_id, r.resident_id, p.property_name, p.owner, u.unit_number, u.unit_status, u.occupied, u.rent, r.first_name, r.last_name, r.move_in_date, r.move_out_date, CASE WHEN r.move_out_date IS NULL THEN 1 ELSE 0 END
FROM properties p
JOIN units u ON p.property_id = u.property_id
JOIN residents r ON u.unit_id = r.unit_id;
"""
execute_query(connection, populate_scd_residents)

# Populate SCD table with vacant units
populate_scd_vacant = """
INSERT INTO scd (property_id, unit_id, resident_id, property_name, owner, unit_number, unit_status, occupied, rent, first_name, last_name, effective_from, effective_to, current_flag)
SELECT p.property_id, u.unit_id, NULL, p.property_name, p.owner, u.unit_number, u.unit_status, u.occupied, u.rent, NULL, NULL, '2023-01-01', NULL, 1
FROM properties p
JOIN units u ON p.property_id = u.property_id
LEFT JOIN residents r ON u.unit_id = r.unit_id
WHERE r.resident_id IS NULL;
"""
execute_query(connection, populate_scd_vacant)

The error 'NOT NULL constraint failed: scd.date' occurred
The error 'NOT NULL constraint failed: scd.date' occurred


In [29]:
get_scd = """SELECT * FROM scd
ORDER BY date, property_id, unit_id;
"""
execute_query(connection, get_scd)

Query executed successfully


In [28]:
cursor = connection.cursor()
rows = cursor.execute(get_scd).fetchall()
print("row count:", len(rows))
for r in rows:
    print(r)

row count: 0


In [ ]:
# revenue = sum of rents where occupied = true


In [ ]:
# A KPI API that retrieves move-ins, move-outs, and occupancy rates by month, within a given start and end date.

# Occupancy rate = (sum of days occupied per unit) / (total units * days in month).
# For example, in a 30 day month, if a property has 50 units and 39 are occupied for 30 days 
# and 2 are occupied for the 15 days, we expect an occupancy rate of ((39 * 30) + (2 * 15)) / (50 * 30) = 0.8